In [5]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.signal import welch

In [6]:
DATA_ROOT = Path(r"C:\Users\Sanjeev\OneDrive\Desktop\Brain ID\EEG\ds004148")
PROCESSED_ROOT = DATA_ROOT / "processed_windows"

FS = 500.0

CHANNELS = ["Fp1", "Fp2", "Fz"]

BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 45),
}

In [7]:
def extract_bandpower_features(window, sfreq):
    """
    Extract relative bandpower features from one EEG window.

    Parameters
    ----------
    window : ndarray
        Shape: (3, samples)
        Channels = Fp1, Fp2, Fz

    sfreq : float
        Sampling frequency.

    Returns
    -------
    features : dict
        15 bandpower features.
    """

    features = {}

    for ch_idx, channel in enumerate(CHANNELS):

        signal = window[ch_idx]

        # Welch PSD
        freqs, psd = welch(
            signal,
            fs=sfreq,
            nperseg=min(500, len(signal)),
            noverlap=250
        )

        # Total power from 1–45 Hz
        total_mask = (
            (freqs >= 1) &
            (freqs <= 45)
        )

        total_power = np.trapezoid(
            psd[total_mask],
            freqs[total_mask]
        )

        for band_name, (low, high) in BANDS.items():

            band_mask = (
                (freqs >= low) &
                (freqs < high)
            )

            band_power = np.trapezoid(
                psd[band_mask],
                freqs[band_mask]
            )

            # Relative power
            relative_power = (
                band_power / total_power
                if total_power > 0
                else 0
            )

            features[
                f"{channel}_{band_name}"
            ] = relative_power

    return features

In [8]:
test_file = sorted(
    PROCESSED_ROOT.glob("*.npz")
)[0]

print(test_file)

C:\Users\Sanjeev\OneDrive\Desktop\Brain ID\EEG\ds004148\processed_windows\sub-01_ses-session1.npz


In [9]:
data = np.load(test_file)

windows = data["windows"]
sfreq = float(data["sfreq"])

print("Windows:", windows.shape)
print("Sampling frequency:", sfreq)

Windows: (69, 3, 2000)
Sampling frequency: 500.0


In [10]:
test_features = extract_bandpower_features(
    windows[0],
    sfreq
)

test_features

{'Fp1_delta': np.float64(0.689025200084337),
 'Fp1_theta': np.float64(0.08004473061010335),
 'Fp1_alpha': np.float64(0.035802695577928284),
 'Fp1_beta': np.float64(0.06151618366061231),
 'Fp1_gamma': np.float64(0.03931049133068305),
 'Fp2_delta': np.float64(0.6283898803854779),
 'Fp2_theta': np.float64(0.10169480556346756),
 'Fp2_alpha': np.float64(0.03298640229957219),
 'Fp2_beta': np.float64(0.08867022898894655),
 'Fp2_gamma': np.float64(0.058096296245309084),
 'Fz_delta': np.float64(0.4896555998370761),
 'Fz_theta': np.float64(0.13353000290287767),
 'Fz_alpha': np.float64(0.08781859066934466),
 'Fz_beta': np.float64(0.0923534357690172),
 'Fz_gamma': np.float64(0.059265153299789694)}

In [11]:
feature_rows = []

npz_files = sorted(
    PROCESSED_ROOT.glob("*.npz")
)

print("Processed recordings:", len(npz_files))

Processed recordings: 180


In [12]:
for file in npz_files:

    # Example:
    # sub-01_ses-session1.npz

    stem = file.stem

    parts = stem.split("_")

    subject = parts[0]
    session = parts[1]

    data = np.load(file)

    windows = data["windows"]
    sfreq = float(data["sfreq"])

    for window_idx, window in enumerate(windows):

        features = extract_bandpower_features(
            window,
            sfreq
        )

        row = {
            "subject": subject,
            "session": session,
            "window": window_idx,
        }

        row.update(features)

        feature_rows.append(row)

In [13]:
features_df = pd.DataFrame(feature_rows)

print(
    "Feature matrix shape:",
    features_df.shape
)

Feature matrix shape: (8735, 18)


In [14]:
features_df.head()

,subject,session,window,Fp1_delta,Fp1_theta,Fp1_alpha,Fp1_beta,Fp1_gamma,Fp2_delta,Fp2_theta,Fp2_alpha,Fp2_beta,Fp2_gamma,Fz_delta,Fz_theta,Fz_alpha,Fz_beta,Fz_gamma
0,sub-01,ses-session1,0,0.689025,0.080045,0.035803,0.061516,0.039310,0.628390,0.101695,0.032986,0.088670,0.058096,0.489656,0.133530,0.087819,0.092353,0.059265
1,sub-01,ses-session1,1,0.802907,0.054172,0.022807,0.033051,0.020225,0.778655,0.055072,0.031630,0.041308,0.035124,0.477474,0.142174,0.102832,0.086347,0.035529
2,sub-01,ses-session1,2,0.335598,0.153675,0.134009,0.109567,0.044424,0.469612,0.111364,0.100917,0.117191,0.047921,0.382444,0.200647,0.117258,0.084152,0.036514
3,sub-01,ses-session1,3,0.576905,0.081624,0.054894,0.096148,0.030582,0.511625,0.116424,0.053488,0.120168,0.055385,0.466996,0.153673,0.069446,0.128968,0.051475
4,sub-01,ses-session1,4,0.575050,0.121707,0.035475,0.052500,0.019073,0.520047,0.111045,0.046061,0.075220,0.034081,0.405714,0.208751,0.055633,0.099451,0.031465


In [15]:
FEATURE_FILE = (
    PROCESSED_ROOT /
    "bandpower_features.csv"
)

features_df.to_csv(
    FEATURE_FILE,
    index=False
)

print(
    "Saved:",
    FEATURE_FILE
)

Saved: C:\Users\Sanjeev\OneDrive\Desktop\Brain ID\EEG\ds004148\processed_windows\bandpower_features.csv
